# 30 — Newhaven Radius Framework

Builds 5, 10, 20, and 30 mile radius analysis framework around Newhaven ERF.

In [ ]:

CENTER_NAME = "Newhaven ERF"
CENTER_LAT = 50.80208
CENTER_LON = 0.04961
DATE_FROM = "2024-01-01"
DATE_TO   = "2025-12-31"
RADII_MILES = [5, 10, 20, 30]
OUTPUT_DIR = "outputs/newhaven_radius"


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, math, os
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon, "start_date": start_date, "end_date": end_date, "hourly": hourly_vars, "timezone": "UTC"}
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:

rows=[]
for miles in RADII_MILES:
    km=miles*1.609344
    lat_deg=km/111.32
    lon_deg=km/(111.32*math.cos(math.radians(CENTER_LAT)))
    rows.append({"center_name":CENTER_NAME,"center_lat":CENTER_LAT,"center_lon":CENTER_LON,"radius_miles":miles,"radius_km":km,"buffer_lat_deg":lat_deg,"buffer_lon_deg":lon_deg,"min_lat":CENTER_LAT-lat_deg,"max_lat":CENTER_LAT+lat_deg,"min_lon":CENTER_LON-lon_deg,"max_lon":CENTER_LON+lon_deg})
framework=pd.DataFrame(rows)
framework_csv=Path(OUTPUT_DIR)/"radius_framework.csv"
framework.to_csv(framework_csv,index=False)
display(framework)


In [ ]:

manifest={"created_utc":datetime.now(timezone.utc).isoformat(),"center_name":CENTER_NAME,"center_lat":CENTER_LAT,"center_lon":CENTER_LON,"date_from":DATE_FROM,"date_to":DATE_TO,"radii_miles":RADII_MILES,"outputs":{"framework_csv":str(framework_csv)}}
manifest_path=Path(OUTPUT_DIR)/"radius_framework_manifest.json"
manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print("Saved:",framework_csv)
print("Saved:",manifest_path)
